# 01 — Data Cleaning


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import numpy as np
import json
from scipy import stats

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)
ninja_log = pd.read_csv('../data/raw/ninja_log.csv')

%matplotlib inline
sns.set_theme(style='whitegrid')

## Missing Data Analysis

Identify targets with missing compile times (INTERFACE, UTILITY, imported). Set generated file git fields to 0/null.


In [ ]:
# Null heatmap overview
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, df) in zip(axes, [('file_metrics', file_df), ('target_metrics', target_df), ('edge_list', edge_df)]):
    sns.heatmap(df.isnull(), ax=ax, cbar=False, yticklabels=False)
    ax.set_title(f'{name} ({df.shape[0]} rows x {df.shape[1]} cols)')
plt.tight_layout()
plt.show()

for name, df in [('file_metrics', file_df), ('target_metrics', target_df), ('edge_list', edge_df)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f'\n{name} — columns with nulls:')
        for col, count in nulls.items():
            print(f'  {col}: {count}/{len(df)}')

In [ ]:
# Categorise missing compile times in file_df
# Files with language=NaN are non-compilable File API entries (.rule, .h, .o refs) — not real missing data
file_df['is_compilable'] = file_df['language'].notna()

non_compilable = file_df[~file_df['is_compilable']]
compilable = file_df[file_df['is_compilable']]
print(f'Compilable files: {len(compilable)}')
print(f'Non-compilable entries (language=NaN): {len(non_compilable)}')
print()
display(non_compilable[['source_file', 'cmake_target', 'is_generated', 'language', 'compile_time_ms']])

In [ ]:
# INTERFACE/UTILITY/imported targets — should have zero compile time, not missing
EXCLUDED_TYPES = {'INTERFACE_LIBRARY', 'UTILITY'}
target_df['is_imported'] = target_df['cmake_target'].str.contains('::', regex=False)
target_df['is_analysis_target'] = (
    ~target_df['target_type'].isin(EXCLUDED_TYPES) &
    ~target_df['is_imported']
)

non_analysis = target_df[~target_df['is_analysis_target']]
print(f'Analysis targets: {target_df["is_analysis_target"].sum()}')
print(f'Non-analysis targets: {len(non_analysis)} (INTERFACE/UTILITY/imported)')
print()
display(non_analysis[['cmake_target', 'target_type', 'compile_time_sum_ms', 'total_build_time_ms']])

In [ ]:
# Confirm generated file git fields are correctly zeroed — not treated as missing
gen = file_df[file_df['is_generated']]
assert (gen['git_commit_count'] == 0).all(), 'Generated files with unexpected git history'
assert gen['git_last_change_date'].isna().all(), 'Generated files with unexpected git dates'
print(f'All {len(gen)} generated files correctly have git_commit_count=0 and NaN git_last_change_date.')
print('These NaN dates are intentional — generated files have no git history.')

## Outlier Detection

Flag files with anomalous compile times using IQR/Z-score. Compare ftime-report totals against ninja log wall-clock times.


In [ ]:
# IQR outlier detection on compile_time_ms (compilable files only)
ct = file_df.loc[file_df['is_compilable'], 'compile_time_ms'].dropna()
q1, q3 = ct.quantile(0.25), ct.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
lower_fence = max(0, q1 - 1.5 * iqr)

file_df['compile_time_outlier_iqr'] = (
    file_df['compile_time_ms'].notna() &
    ((file_df['compile_time_ms'] > upper_fence) | (file_df['compile_time_ms'] < lower_fence))
)

print(f'IQR fences: [{lower_fence:.0f}ms, {upper_fence:.0f}ms]')
print(f'IQR outliers: {file_df["compile_time_outlier_iqr"].sum()}')
outliers_iqr = file_df[file_df['compile_time_outlier_iqr']]
if not outliers_iqr.empty:
    display(outliers_iqr[['source_file', 'cmake_target', 'compile_time_ms', 'is_generated']])

In [ ]:
# Z-score outlier detection
z_scores = pd.Series(np.abs(stats.zscore(ct)), index=ct.index)
file_df['compile_time_outlier_zscore'] = file_df.index.isin(z_scores[z_scores > 2].index)

both = file_df['compile_time_outlier_iqr'] & file_df['compile_time_outlier_zscore']
print(f'IQR outliers:     {file_df["compile_time_outlier_iqr"].sum()}')
print(f'Z-score outliers: {file_df["compile_time_outlier_zscore"].sum()}')
print(f'Flagged by both:  {both.sum()}')

In [ ]:
# Visualise compile time outliers
subset = file_df[file_df['is_compilable']].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=subset, x='compile_time_ms', ax=ax1, color='lightblue', width=0.4)
sns.stripplot(data=subset, x='compile_time_ms', hue='compile_time_outlier_iqr',
              palette={True: 'red', False: 'steelblue'}, ax=ax1, jitter=0.3, legend=True)
ax1.set_title('Compile Time Distribution (red = IQR outlier)')
ax1.set_xlabel('compile_time_ms')

sns.boxplot(data=subset, x='compile_time_ms', y='is_generated', orient='h', ax=ax2)
ax2.set_title('Compile Time: Authored vs Generated')
ax2.set_ylabel('is_generated')

plt.tight_layout()
plt.show()

In [ ]:
# -ftime-report vs ninja wall-clock consistency check
ninja_compile = ninja_log[ninja_log['step_type'] == 'compile'][['source_file', 'cmake_target', 'duration_ms']]

consistency = file_df[file_df['is_compilable']].merge(
    ninja_compile.rename(columns={'duration_ms': 'ninja_ms'}),
    on=['source_file', 'cmake_target'],
    how='left',
)
consistency['time_diff_ms'] = (consistency['compile_time_ms'] - consistency['ninja_ms']).abs()
consistency['time_diff_pct'] = consistency['time_diff_ms'] / consistency['ninja_ms'].replace(0, np.nan) * 100

inconsistent = consistency[consistency['time_diff_pct'] > 10]
print(f'Files with >10% discrepancy between compile_time_ms and ninja wall-clock: {len(inconsistent)}')
if inconsistent.empty:
    print('All compile times are consistent with ninja log wall-clock times.')
else:
    display(inconsistent[['source_file', 'compile_time_ms', 'ninja_ms', 'time_diff_pct']])

# GCC phase sum check (only if ftime-report data is present)
gcc_cols = [c for c in file_df.columns if c.startswith('gcc_') and c != 'gcc_total_time_ms']
if file_df[gcc_cols].notna().any().any():
    phase_sum = file_df[gcc_cols].sum(axis=1)
    diff = (phase_sum - file_df['gcc_total_time_ms']).abs()
    print(f'\nGCC phase sum vs gcc_total_time_ms max diff: {diff.max():.2f}ms')
else:
    print('\nNo -ftime-report data present — skipping GCC phase sum consistency check.')

## Path Alignment Validation

Verify all file paths across collectors match canonical paths from File API.


In [ ]:
# File API canonical paths vs file_df
with open('../data/raw/cmake_file_api/files.json') as f:
    file_api_entries = json.load(f)
file_api_paths = {e['path'] for e in file_api_entries}
file_df_paths = set(file_df['source_file'])

in_df_not_api = file_df_paths - file_api_paths
in_api_not_df = file_api_paths - file_df_paths

print(f'file_df paths:         {len(file_df_paths)}')
print(f'File API canonical:    {len(file_api_paths)}')
print(f'In file_df, not API:   {len(in_df_not_api)}')
print(f'In API, not file_df:   {len(in_api_not_df)}')

if in_df_not_api:
    print('\nPATH MISMATCH — in file_df but not File API:')
    for p in sorted(in_df_not_api):
        print(f'  {p}')
if in_api_not_df:
    print('\nPATH MISMATCH — in File API but not file_df:')
    for p in sorted(in_api_not_df):
        print(f'  {p}')
if not in_df_not_api and not in_api_not_df:
    print('\nAll file_df paths match File API canonical paths.')

In [ ]:
# Ninja compile paths vs file_df
ninja_compiled_paths = set(ninja_log[ninja_log['step_type'] == 'compile']['source_file'].dropna())
authored_df_paths = set(file_df[~file_df['is_generated']]['source_file'])
generated_df_paths = set(file_df[file_df['is_generated']]['source_file'])

authored_not_ninja = authored_df_paths - ninja_compiled_paths
ninja_not_accounted = ninja_compiled_paths - authored_df_paths - generated_df_paths

print(f'Authored files in file_df:  {len(authored_df_paths)}')
print(f'Generated files in file_df: {len(generated_df_paths)}')
print(f'Compile steps in ninja:     {len(ninja_compiled_paths)}')
print(f'Authored in file_df but not in ninja: {len(authored_not_ninja)}')
print(f'Ninja compiles not accounted for:     {len(ninja_not_accounted)}')

if authored_not_ninja:
    print('\nAuthored files missing from ninja compile log:')
    for p in sorted(authored_not_ninja):
        print(f'  {p}')
if ninja_not_accounted:
    print('\nNinja compile steps not in file_df (authored or generated):')
    for p in sorted(ninja_not_accounted):
        print(f'  {p}')

## Target Validation

Cross-reference File API targets with ninja log targets.


In [ ]:
# File API targets vs ninja targets
with open('../data/raw/cmake_file_api/targets.json') as f:
    api_targets = json.load(f)
api_target_names = {t['name'] for t in api_targets}
api_target_types = {t['name']: t['type'] for t in api_targets}

ninja_target_names = set(ninja_log['cmake_target'].dropna().unique())

in_api_not_ninja = api_target_names - ninja_target_names
in_ninja_not_api = ninja_target_names - api_target_names

print(f'File API targets: {len(api_target_names)}')
print(f'Ninja targets:    {len(ninja_target_names)}')
print()
if in_api_not_ninja:
    print('Targets in File API but not built by ninja (expected for INTERFACE/UTILITY/imported):')
    for t in sorted(in_api_not_ninja):
        print(f'  {t} ({api_target_types.get(t, "unknown")})')
print()
if in_ninja_not_api:
    print('UNEXPECTED: targets built by ninja not in File API:')
    for t in sorted(in_ninja_not_api):
        print(f'  {t}')
else:
    print('No ninja targets are missing from the File API.')

In [ ]:
# Graph nodes vs target_df
graph_nodes = set(G.nodes())
target_df_names = set(target_df['cmake_target'])

in_graph_not_df = graph_nodes - target_df_names
in_df_not_graph = target_df_names - graph_nodes

print(f'Graph nodes:    {len(graph_nodes)}')
print(f'target_df rows: {len(target_df)}')
print(f'In graph but not target_df: {in_graph_not_df or "none"}')
print(f'In target_df but not graph: {in_df_not_graph or "none"}')

# Flag analysis targets with files but zero build time
suspicious = target_df[
    target_df['is_analysis_target']
    & (target_df['total_build_time_ms'] == 0)
    & (target_df['file_count'] > 0)
]
if not suspicious.empty:
    print('\nSuspicious: analysis targets with file_count>0 but zero total_build_time_ms:')
    display(suspicious[['cmake_target', 'target_type', 'file_count', 'total_build_time_ms']])
else:
    print('\nNo suspicious zero-build-time analysis targets.')

## Codegen Validation

Verify is_generated files have no git history (or flag anomalies).


In [ ]:
# Codegen inventory coverage
with open('../data/raw/cmake_file_api/codegen_inventory.json') as f:
    codegen_inv = json.load(f)

inventory_files = {f for files in codegen_inv.values() for f in files}
inventory_targets = set(codegen_inv.keys())

gen_files_df = set(file_df[file_df['is_generated']]['source_file'])
gen_targets_df = set(file_df[file_df['is_generated']]['cmake_target'])

print(f'Codegen inventory: {len(inventory_targets)} targets, {len(inventory_files)} files')
print(f'is_generated in file_df: {len(gen_files_df)} files from {len(gen_targets_df)} targets')
print()

in_inv_not_df = inventory_files - gen_files_df
in_df_not_inv = gen_files_df - inventory_files
print(f'In inventory but not in file_df (is_generated): {len(in_inv_not_df)}')
print(f'In file_df (is_generated) but not in inventory: {len(in_df_not_inv)}')
if in_inv_not_df:
    for p in sorted(in_inv_not_df):
        print(f'  missing from file_df: {p}')
if in_df_not_inv:
    for p in sorted(in_df_not_inv):
        print(f'  missing from inventory: {p}')

In [ ]:
# Verify generated files have no git history (or are in source tree — rare case)
source_dir = str(cfg.source_dir)
gen_df = file_df[file_df['is_generated']].copy()

gen_df['has_no_git_history'] = (gen_df['git_commit_count'] == 0) & gen_df['git_last_change_date'].isna()
gen_df['in_source_tree'] = gen_df['source_file'].str.startswith(source_dir)
gen_df['codegen_valid'] = gen_df['has_no_git_history'] | gen_df['in_source_tree']

anomalous = gen_df[~gen_df['codegen_valid']]
print(f'Generated files: {len(gen_df)}')
print(f'  No git history (expected):     {gen_df["has_no_git_history"].sum()}')
print(f'  In source tree (rare case):    {gen_df["in_source_tree"].sum()}')
print(f'  ANOMALOUS (neither condition): {len(anomalous)}')

if not anomalous.empty:
    display(anomalous[['source_file', 'cmake_target', 'git_commit_count', 'git_last_change_date', 'in_source_tree']])

In [ ]:
# Generated vs authored file distribution per target (compilable files only)
counts = file_df[file_df['is_compilable']].groupby(['cmake_target', 'is_generated']).size().unstack(fill_value=0)
counts.columns = ['authored', 'generated']
counts = counts[counts.sum(axis=1) > 0].sort_values('generated', ascending=False)

ax = counts.plot(kind='bar', stacked=True, figsize=(12, 4), color=['steelblue', 'orange'])
ax.set_title('Files per target: authored vs generated (compilable only)')
ax.set_xlabel('Target')
ax.set_ylabel('File count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Type Casting and Normalisation

Ensure correct types, normalise time units to milliseconds.


In [ ]:
from build_optimiser.metrics import FILE_METRICS_SCHEMA, TARGET_METRICS_SCHEMA, EDGE_LIST_SCHEMA

# Dtype audit: compare actual pandas dtypes against PyArrow schema
def dtype_audit(df, schema, label):
    mismatches = []
    for field in schema:
        if field.name not in df.columns:
            mismatches.append((field.name, 'MISSING', str(field.type)))
            continue
        actual = str(df[field.name].dtype)
        expected = str(field.type)
        if actual != expected and actual not in ('object', 'str', expected):
            mismatches.append((field.name, actual, expected))
    if mismatches:
        print(f'\n{label} — dtype mismatches:')
        for col, actual, expected in mismatches:
            print(f'  {col}: actual={actual}, schema={expected}')
    else:
        print(f'\n{label} — all dtypes match schema.')

dtype_audit(file_df, FILE_METRICS_SCHEMA, 'file_metrics')
dtype_audit(target_df, TARGET_METRICS_SCHEMA, 'target_metrics')
dtype_audit(edge_df, EDGE_LIST_SCHEMA, 'edge_list')

In [ ]:
# Verify time units are milliseconds (value ranges should be plausible)
time_cols_file = ['compile_time_ms', 'gcc_parse_time_ms', 'gcc_template_instantiation_ms',
                  'gcc_codegen_time_ms', 'gcc_optimization_time_ms', 'gcc_total_time_ms']
time_cols_target = ['compile_time_sum_ms', 'compile_time_max_ms', 'total_build_time_ms',
                    'codegen_time_ms', 'archive_time_ms', 'link_time_ms']

print('file_df time column ranges:')
for col in time_cols_file:
    if col in file_df.columns:
        vals = file_df[col].dropna()
        if not vals.empty:
            print(f'  {col}: min={vals.min():.0f}, max={vals.max():.0f}, mean={vals.mean():.0f}')
        else:
            print(f'  {col}: all null (no data)')

print('\ntarget_df time column ranges:')
for col in time_cols_target:
    if col in target_df.columns:
        vals = target_df[col].dropna()
        if not vals.empty:
            print(f'  {col}: min={vals.min():.0f}, max={vals.max():.0f}, sum={vals.sum():.0f}')

In [ ]:
# Apply type casts: float64 -> Int64 (nullable integer) for schema-declared int64 columns with NaN
float_to_int_cols = [
    'compile_time_ms', 'header_max_depth', 'unique_headers',
    'total_includes', 'preprocessed_bytes', 'object_size_bytes',
]
for col in float_to_int_cols:
    if col in file_df.columns and file_df[col].dtype == 'float64':
        file_df[col] = file_df[col].astype(pd.Int64Dtype())

# Normalise language column (consolidate 'C++' -> 'CXX' if present)
if 'language' in file_df.columns:
    file_df['language'] = file_df['language'].replace({'C++': 'CXX'})

print('Type casts applied to file_df.')
print(file_df[float_to_int_cols].dtypes)
print(f'\nNulls preserved: {file_df["compile_time_ms"].isna().sum()} NaN rows in compile_time_ms')
print(f'\nlanguage values: {file_df["language"].value_counts(dropna=False).to_dict()}')

In [ ]:
# Final null summary
def null_summary(df, label, expected_null_cols=None):
    expected_null_cols = expected_null_cols or []
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f'{label}: no nulls remaining')
        return
    print(f'\n{label} — remaining nulls:')
    for col, count in nulls.items():
        tag = '(expected)' if col in expected_null_cols else '** REVIEW **'
        print(f'  {col}: {count} {tag}')

gcc_cols = [c for c in file_df.columns if c.startswith('gcc_')]
file_expected_nulls = gcc_cols + [
    'language', 'compile_time_ms', 'header_max_depth', 'unique_headers',
    'total_includes', 'preprocessed_bytes', 'object_size_bytes',
    'git_last_change_date', 'expansion_ratio', 'compile_rate_lines_per_sec',
    'object_efficiency', 'code_lines', 'blank_lines', 'comment_lines',
    'source_size_bytes',
]
null_summary(file_df, 'file_df', expected_null_cols=file_expected_nulls)
null_summary(target_df, 'target_df')
null_summary(edge_df, 'edge_df', expected_null_cols=['from_dependency'])

## Write Cleaned Data

Output cleaned versions of all three Parquet files.


In [ ]:
# Prepare clean DataFrames — drop transient analysis columns
transient_file_cols = ['is_compilable', 'compile_time_outlier_iqr', 'compile_time_outlier_zscore']
transient_target_cols = ['is_imported', 'is_analysis_target']

file_df_clean = file_df.drop(columns=[c for c in transient_file_cols if c in file_df.columns])
target_df_clean = target_df.drop(columns=[c for c in transient_target_cols if c in target_df.columns])
edge_df_clean = edge_df.copy()

print(f'file_df_clean:   {file_df_clean.shape[0]} rows, {file_df_clean.shape[1]} columns')
print(f'target_df_clean: {target_df_clean.shape[0]} rows, {target_df_clean.shape[1]} columns')
print(f'edge_df_clean:   {edge_df_clean.shape[0]} rows, {edge_df_clean.shape[1]} columns')

In [ ]:
# Write cleaned parquet files with explicit schemas
OUT_DIR = Path('../data/processed')

file_table = pa.Table.from_pandas(file_df_clean, schema=FILE_METRICS_SCHEMA, safe=False)
pq.write_table(file_table, OUT_DIR / 'file_metrics.parquet', compression='snappy')

target_table = pa.Table.from_pandas(target_df_clean, schema=TARGET_METRICS_SCHEMA, safe=False)
pq.write_table(target_table, OUT_DIR / 'target_metrics.parquet', compression='snappy')

edge_table = pa.Table.from_pandas(edge_df_clean, schema=EDGE_LIST_SCHEMA, safe=False)
pq.write_table(edge_table, OUT_DIR / 'edge_list.parquet', compression='snappy')

print('Wrote cleaned parquet files to data/processed/')

In [ ]:
# Round-trip verification
verify_file = pd.read_parquet(OUT_DIR / 'file_metrics.parquet')
verify_target = pd.read_parquet(OUT_DIR / 'target_metrics.parquet')
verify_edge = pd.read_parquet(OUT_DIR / 'edge_list.parquet')

for name, orig, reloaded in [
    ('file_metrics', file_df_clean, verify_file),
    ('target_metrics', target_df_clean, verify_target),
    ('edge_list', edge_df_clean, verify_edge),
]:
    assert orig.shape[0] == reloaded.shape[0], f'{name}: row count mismatch'
    assert set(orig.columns) == set(reloaded.columns), f'{name}: column mismatch'
    print(f'{name}: {reloaded.shape[0]} rows, {reloaded.shape[1]} columns — OK')

print('\nAll cleaned parquet files written and verified.')